In [ ]:
%pip install neo4j python-dotenv pandas

import os
from neo4j import GraphDatabase
import pandas as pd
import random
from dotenv import load_dotenv

# Load environment variables
load_dotenv('/home/mindflayer/Projects/Personalization/.env_db')

# Debug: print environment variables
print("VPS_IP:", os.getenv('VPS_IP'))
print("NEO4J_USER:", os.getenv('NEO4J_USER'))
print("NEO4J_PASSWORD:", "***")  # Don't print password

# Neo4j connection details
NEO4J_URI = f"bolt://{os.getenv('VPS_IP')}:7687"
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

print(f"Connecting to Neo4j at {NEO4J_URI}")

In [6]:
# Connect to Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def test_connection():
    with driver.session() as session:
        result = session.run("MATCH () RETURN count(*) as node_count")
        record = result.single()
        print(f"Connected successfully! Database has {record['node_count']} nodes.")

test_connection()

Connected successfully! Database has 48579 nodes.


In [9]:
# Get random sample of nodes (updated to use elementId)
def get_random_nodes(sample_size=1000):
    with driver.session() as session:
        # First get total node count
        result = session.run("MATCH (n) RETURN count(n) as total_nodes")
        total_nodes = result.single()['total_nodes']
        print(f"Total nodes in database: {total_nodes}")

        # If we have fewer nodes than requested, return all
        actual_sample_size = min(sample_size, total_nodes)
        print(f"Sampling {actual_sample_size} nodes...")

        # Get random nodes using SKIP and LIMIT with random ordering
        query = f"""
        MATCH (n)
        RETURN elementId(n) as node_id, labels(n) as labels, properties(n) as properties
        ORDER BY rand()
        LIMIT {actual_sample_size}
        """

        result = session.run(query)
        nodes = []
        for record in result:
            nodes.append({
                'node_id': record['node_id'],
                'labels': record['labels'],
                'properties': record['properties']
            })

        return nodes

# Get sample nodes
sample_nodes = get_random_nodes(1000)
print(f"Retrieved {len(sample_nodes)} nodes")
print("Sample node:", sample_nodes[0] if sample_nodes else "No nodes found")

Total nodes in database: 48579
Sampling 1000 nodes...
Retrieved 1000 nodes
Sample node: {'node_id': '4:a7055092-7933-4d6f-a93a-a08abf5e63aa:13489', 'labels': ['Paper'], 'properties': {'summary': 'OBJECTIVE:  \nShared e-mobility systems face significant fleet rebalancing challenges due to electric vehicle (EV) range limitations, long charging times, and dynamically expanding station networks. Existing rebalancing strategies fail to account for non-stationary action spaces caused by continuous station deployment and closure, and do not integrate EV-specific constraints like battery range and charging dynamics. This work addresses the gap by modeling rebalancing as a multi-agent reinforcement learning (MARL) problem that explicitly incorporates EV range, charging time, and dynamic infrastructure expansion.\n\nMETHODOLOGY:  \nA high-fidelity simulator is developed and calibrated using one year of real-world operational data from a Shanghai-based shared e-mobility system, including 7 millio

In [ ]:
# Get references (relationships) for nodes - optimized version
def get_all_node_references(node_ids):
    with driver.session() as session:
        # Convert node_ids to list for Cypher IN clause
        node_id_list = [node['node_id'] for node in node_ids]

        # Single query to get all relationships for all sampled nodes
        query = """
        MATCH (n)-[r]-(m)
        WHERE elementId(n) IN $node_ids
        RETURN elementId(n) as source_node_id, type(r) as relationship_type,
               elementId(m) as target_node_id, labels(m) as target_labels,
               properties(m) as target_properties,
               CASE WHEN startNode(r) = n THEN 'outgoing' ELSE 'incoming' END as direction
        """

        result = session.run(query, node_ids=node_id_list)
        references = []
        for record in result:
            references.append({
                'source_node_id': record['source_node_id'],
                'relationship_type': record['relationship_type'],
                'target_node_id': record['target_node_id'],
                'target_labels': record['target_labels'],
                'target_properties': record['target_properties'],
                'direction': record['direction']
            })

        return references

# Get references for all sample nodes in one query
print("Getting references for sample nodes...")
all_references = get_all_node_references(sample_nodes)

print(f"Total references found: {len(all_references)}")

Getting references for sample nodes...


In [ ]:
# Save results to CSV
def save_to_csv(nodes, references, filename="neo4j_sample_references.csv"):
    # Create DataFrame for nodes
    nodes_df = pd.DataFrame(nodes)

    # Create DataFrame for references
    references_df = pd.DataFrame(references)

    # Save to CSV
    nodes_df.to_csv(f"nodes_{filename}", index=False)
    references_df.to_csv(filename, index=False)

    print(f"Saved {len(nodes)} nodes to nodes_{filename}")
    print(f"Saved {len(references)} references to {filename}")

    return nodes_df, references_df

# Save the results
nodes_df, references_df = save_to_csv(sample_nodes, all_references)

# Display summary
print("\nSummary:")
print(f"Nodes sampled: {len(sample_nodes)}")
print(f"Total references: {len(all_references)}")
print(f"Average references per node: {len(all_references)/len(sample_nodes):.2f}")

# Close the driver
driver.close()
print("Neo4j connection closed.")